In [12]:
from __future__ import annotations

import duckdb
import pandas as pd

from lake_client import LAKE_SOURCE, describe, list_tables, load_table
from etl import config as C

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

describe()

LAKE_SOURCE = 's3'  ->  online (remote URIs from .env)
target_uri  = s3://loka-test-poc-201302614689-us-east-1

  bronze       7 table(s): bronze_device_metadata, bronze_heart_rate, bronze_participants, bronze_sleep_sessions, bronze_sleep_stages, bronze_steps, bronze_wellness_survey
  silver       8 table(s): silver_devices, silver_heart_rate_minute, silver_participant_day_coverage, silver_participants, silver_sleep_sessions, silver_sleep_stages, silver_steps_minute, silver_wellness_survey
  gold         5 table(s): _analytics, gold_chronotype_cohort_metrics, gold_daily_participant_metrics, gold_participant_devices, gold_weekly_participant_summary
  _quarantine  2 table(s): sleep_sessions, wellness_survey
  _dq          1 table(s): dq_checks


## Bronze — verbatim + lineage

Nothing is cleaned here. Timestamps are still strings, sentinels are still
in the data, the undocumented unit field (if present) is still whatever it
was called in the export. The three lineage columns show where every row
came from and which run landed it.

In [13]:
bronze_steps = load_table(C.BRONZE, "bronze_steps")
print(bronze_steps.shape)
bronze_steps.head()

(2014950, 8)


,timestamp,steps,participant_id,_source_uri,_ingested_at,_run_id,_unit,_source_participant_dir
0,2026-01-08T00:00:00,0,P001,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z,<NA>,P001
1,2026-01-08T00:01:00,0,P001,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z,<NA>,P001
2,2026-01-08T00:02:00,2,P001,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z,<NA>,P001
3,2026-01-08T00:03:00,0,P001,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z,<NA>,P001
4,2026-01-08T00:04:00,0,P001,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z,<NA>,P001


In [14]:
bronze_steps[["_source_uri", "_ingested_at", "_run_id"]].drop_duplicates().head()

,_source_uri,_ingested_at,_run_id
0,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z
40320,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z
80640,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z
120960,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z
161280,s3://de-tech-assessment-396587179375-us-east-1...,2026-09-22 00:23:00.394958,run_20260922T002259Z


## Silver — cleaned, typed, flagged

Every correction leaves a boolean flag column behind rather than
disappearing silently — `was_rescaled`, `was_sentinel`, `is_stuck`, etc.

In [15]:
silver_hr = load_table(C.SILVER, "silver_heart_rate_minute")
print(silver_hr.shape)
silver_hr.head()

(2009160, 8)


,participant_id,timestamp,heart_rate_bpm,is_stuck,was_utc,was_off_minute,was_out_of_range,date
0,P001,2026-01-08 00:00:00,59,False,False,False,False,2026-01-08
1,P001,2026-01-08 00:01:00,58,False,False,False,False,2026-01-08
2,P001,2026-01-08 00:02:00,57,False,False,False,False,2026-01-08
3,P001,2026-01-08 00:03:00,57,False,False,False,False,2026-01-08
4,P001,2026-01-08 00:04:00,56,False,False,False,False,2026-01-08


In [16]:
# How many minutes were actually touched by each rule, on this run:
silver_hr[["was_utc", "was_off_minute", "was_out_of_range", "is_stuck"]].sum()

was_utc             2880
was_off_minute        10
was_out_of_range       4
is_stuck             360
dtype: int64

## Quarantine — rejected, not dropped

Sleep sessions and survey rows that couldn't be repaired land here instead
of Silver, with the reason attached, instead of silently vanishing.

In [17]:
if "sleep_sessions" in list_tables(C.QUARANTINE):
    quarantined_sleep = load_table(C.QUARANTINE, "sleep_sessions")
    print(quarantined_sleep.shape)
    quarantined_sleep["reject_reason"].value_counts()
else:
    print("no quarantined sleep sessions on this run")

(7, 22)


## Gold — the four analysis-ready tables

All four are chained SQL views over Silver (see `poc/sql/`), so they are
always consistent with each other on a given run.

In [18]:
gold_tables = {
    name: load_table(C.GOLD, name)
    for name in (
        "gold_participant_devices",
        "gold_daily_participant_metrics",
        "gold_weekly_participant_summary",
        "gold_chronotype_cohort_metrics",
    )
}
for name, df in gold_tables.items():
    print(f"{name:<34} {df.shape}")

gold_participant_devices           (50, 11)
gold_daily_participant_metrics     (1400, 39)
gold_weekly_participant_summary    (250, 39)
gold_chronotype_cohort_metrics     (12, 27)


In [19]:
gold_tables["gold_daily_participant_metrics"].head()

,participant_id,study_week,study_day,chronotype_declared,age,gender,max_heart_rate,steps_total,active_minutes,steps_peak_minute,activity_midpoint_hour,hr_mean_bpm,hr_resting_bpm,hr_min_bpm,hr_max_bpm,total_sleep_min,time_in_bed_min,sleep_efficiency_pct,deep_sleep_pct,rem_sleep_pct,mid_sleep_hour,sleep_onset_hour,restlessness,fatigue_score,stress_score,readiness_score,sleep_quality_score,steps_minutes,hr_minutes,coverage_pct,is_complete_day,is_missing_day,has_sleep_session,has_survey,minutes_nulled,minutes_unit_rescaled,hr_stuck_minutes,minutes_tz_corrected,date
0,P001,1,1,B,38,male,167,8523.0,925,32,14.070,60.3,55.0,49,74,490.0,561.0,87.34,18.78,23.47,2.991,-1.100,0.077604,3,3,8,3,1440,1440,100.0,True,False,True,True,0,0,0,0,2026-01-08
1,P002,1,1,A,31,male,199,6584.0,919,26,14.093,70.3,65.0,61,82,401.0,454.0,88.33,2.99,18.70,2.859,-0.483,0.071804,4,3,7,3,1440,1440,100.0,True,False,True,True,0,0,0,0,2026-01-08
2,P003,1,1,B,51,male,179,8298.0,938,32,14.105,65.7,60.0,55,81,402.0,466.0,86.27,9.20,19.65,3.530,0.167,0.103476,3,2,4,4,1440,1440,100.0,True,False,True,True,0,0,0,0,2026-01-08
3,P004,1,1,A,26,male,188,10275.0,943,40,14.153,67.7,62.0,58,83,309.0,347.0,89.05,10.68,22.98,2.064,-0.517,0.043776,3,2,4,2,1440,1440,100.0,True,False,True,True,0,0,0,0,2026-01-08
4,P005,1,1,A,40,female,193,2695.0,851,11,13.923,104.5,69.0,65,135,455.0,530.0,85.85,11.43,18.90,1.857,-1.950,0.139491,4,4,7,4,1440,1440,100.0,True,False,True,True,0,1440,0,0,2026-01-08


## Data quality — every `Check` emitted on this run

One row per rule per table; `severity` is `error` (a handled source
defect), `warn`, `info`, or `assert_fail` (a pipeline invariant broken —
the thing `--strict` fails the run on).

In [20]:
dq = load_table(C.DQ, "dq_checks")
dq["severity"].value_counts()

severity
assert_pass    20
error          16
warn            3
info            2
Name: count, dtype: int64

In [21]:
dq.sort_values("n_affected", ascending=False)[
    ["table", "rule", "severity", "n_affected", "detail"]
].head(15)

,table,rule,severity,n_affected,detail
18,silver_steps_minute,unit_field_nonstandard_name,warn,120963,"unit arrives as `_unit`, not `unit`; resolved ..."
13,silver_steps_minute,unit_rescaled,error,120963,rows carried an hourly rate on a per-minute gr...
1,silver_heart_rate_minute,duplicate_rows_removed,error,5790,"duplicate ('participant_id', 'timestamp') acro..."
11,silver_steps_minute,duplicate_rows_removed,error,5790,"duplicate ('participant_id', 'timestamp') acro..."
4,silver_heart_rate_minute,timezone_normalised,error,2880,"rows exported as Z-suffixed UTC, pulled back t..."
39,silver_participant_day_coverage,grid_size,info,1400,50 participants x 28 days
17,silver_sleep_sessions,source_efficiency_ignored,warn,1132,1132/1400 sessions report efficiency_pct exact...
2,silver_heart_rate_minute,stuck_sensor_flagged,error,360,2 run(s) of >= 60 identical consecutive minute...
12,silver_steps_minute,timestamp_snapped_to_minute,error,28,seconds != 00; floored to the minute before de...
3,silver_heart_rate_minute,timestamp_snapped_to_minute,error,28,seconds != 00; floored to the minute before de...


## Ad-hoc SQL over what's loaded

DuckDB queries straight over the DataFrames already in this notebook — no
registration step needed, same engine `gold.py` builds the tables with.
`is_complete_day` is the flag to filter on for volume metrics (steps); it
travels with the row rather than being applied at write time, so it's a
query-time decision here too.

In [22]:
daily = gold_tables["gold_daily_participant_metrics"]
devices = gold_tables["gold_participant_devices"]

duckdb.sql(
    """
    SELECT d.wear_site,
           count(*)                          AS participant_days,
           round(avg(m.steps_total), 0)      AS avg_daily_steps,
           round(avg(m.hr_resting_bpm), 1)   AS avg_resting_hr
    FROM daily m
    JOIN devices d USING (participant_id)
    WHERE m.is_complete_day
    GROUP BY 1
    ORDER BY avg_daily_steps DESC
    """
)

┌───────────────────┬──────────────────┬─────────────────┬────────────────┐
│     wear_site     │ participant_days │ avg_daily_steps │ avg_resting_hr │
│      varchar      │      int64       │     double      │     double     │
├───────────────────┼──────────────────┼─────────────────┼────────────────┤
│ wrist_nondominant │              557 │          9092.0 │           64.3 │
│ wrist_dominant    │              836 │          8856.0 │           61.5 │
└───────────────────┴──────────────────┴─────────────────┴────────────────┘

The query above answers one number per group. This one just hands back
rows to actually look at — the ten highest-activity complete participant-
days on record, unaggregated:

In [23]:
duckdb.sql(
    """
    SELECT participant_id, date, steps_total, active_minutes,
           hr_resting_bpm, sleep_efficiency_pct, coverage_pct
    FROM daily
    WHERE is_complete_day
    ORDER BY steps_total DESC
    LIMIT 10
    """
).df()

,participant_id,date,steps_total,active_minutes,hr_resting_bpm,sleep_efficiency_pct,coverage_pct
0,P035,2026-01-21,14749.0,970,63.0,88.64,100.0
1,P035,2026-02-01,14680.0,966,63.0,82.30,100.0
2,P035,2026-01-29,14672.0,966,63.0,83.33,100.0
3,P035,2026-01-30,14663.0,959,62.0,84.10,100.0
4,P035,2026-01-26,14585.0,968,63.0,85.26,100.0
5,P035,2026-01-12,14574.0,957,63.0,83.63,100.0
6,P035,2026-01-24,14562.0,955,63.0,89.98,100.0
7,P035,2026-01-18,14556.0,962,63.0,86.41,100.0
8,P035,2026-01-16,14504.0,947,63.0,83.18,100.0
9,P035,2026-01-10,14490.0,961,52.0,87.07,100.0
